# Gold Layer - Business Metrics & KPIs

This notebook creates business-ready aggregated tables and KPIs from the clean silver layer data.

## Purpose
The gold layer provides pre-aggregated, business-focused metrics optimized for:
- Executive dashboards
- Business intelligence reporting
- Self-service analytics
- Performance monitoring

## Data Source
**Silver Layer Tables:**
- `workspace.silver.fact_sales_denormalized` - Complete sales data with all business context
- `workspace.silver.fact_sales` - Clean sales transactions
- `workspace.silver.dim_*` - Dimension tables

## Gold Layer Metrics

### Core Business Aggregations
1. **Total Sales & Orders** - Overall business performance
2. **Sales by Time** - Monthly trends and seasonality
3. **Sales by Geography** - Country/region performance
4. **Sales by Product** - Product category and SKU analysis

### Advanced KPIs
5. **Customer Summary** - Customer lifetime value, purchase frequency, RFM analysis
6. **Product Performance** - Profit margins, best/worst sellers
7. **Regional Performance** - Geographic insights and trends
8. **Time Series Analysis** - Year-over-year growth, moving averages

In [0]:
# Load clean silver layer data for aggregations

from pyspark.sql.functions import col, sum as spark_sum, count, avg, min, max, round as spark_round
from pyspark.sql.functions import month, year, date_format, countDistinct, when, datediff, current_date

print("=" * 80)
print("LOADING SILVER LAYER DATA")
print("=" * 80)

# Load the denormalized fact table (easiest for aggregations)
fact_sales = spark.table("workspace.silver.fact_sales_denormalized")

print(f"\n✓ Loaded fact_sales_denormalized: {fact_sales.count():,} rows")
print(f"   Columns: {len(fact_sales.columns)}")

print("\nAvailable columns for analysis:")
for col_name in fact_sales.columns:
    print(f"  - {col_name}")

print("\nSample data:")
display(fact_sales.limit(5))

print("\n" + "=" * 80)

LOADING SILVER LAYER DATA

✓ Loaded fact_sales_denormalized: 63,738 rows
   Columns: 28

Available columns for analysis:
  - OrderNumber
  - OrderDateKey
  - OrderDate
  - MonthName
  - CalendarYear
  - CustomerKey
  - CustomerName
  - CustomerGender
  - MaritalStatus
  - GeographyKey
  - City
  - Region
  - Country
  - ProductKey
  - Product_Name
  - Color
  - ProductSubcategoryKey
  - ProductSubcategoryName
  - ProductCategoryKey
  - ProductCategoryName
  - OrderQuantity
  - List_Price
  - Product_Cost
  - StandardCost
  - ListPrice
  - Revenue
  - Cost
  - Profit

Sample data:


OrderNumber,OrderDateKey,OrderDate,MonthName,CalendarYear,CustomerKey,CustomerName,CustomerGender,MaritalStatus,GeographyKey,City,Region,Country,ProductKey,Product_Name,Color,ProductSubcategoryKey,ProductSubcategoryName,ProductCategoryKey,ProductCategoryName,OrderQuantity,List_Price,Product_Cost,StandardCost,ListPrice,Revenue,Cost,Profit
20061722,20050722,2005-07-22,July,2005,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,344,Product 89,Silver,1,Mountain Bikes,1,Bikes,22,3399.99,1912.15,1912.15,3399.99,74799.78,42067.30,32732.48
20081722,20070722,2007-07-22,July,2007,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,353,Product 94,Silver,1,Mountain Bikes,1,Bikes,22,2319.99,1265.62,1265.62,2319.99,51039.78,27843.64,23196.14
20081722,20070722,2007-07-22,July,2007,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,485,Product 163,NA,30,Fenders,4,Accessories,22,21.98,8.22,8.22,21.98,483.56,180.84,302.72
20082104,20071104,2007-11-04,November,2007,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,530,Product 208,NA,37,Tires and Tubes,4,Accessories,4,4.99,1.87,1.87,4.99,19.96,7.48,12.48
20082104,20071104,2007-11-04,November,2007,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,214,Product 3,Red,31,Helmets,4,Accessories,4,34.99,13.09,13.09,34.99,139.96,52.36,87.60


In [0]:
# Aggregate 1: Total Sales & Orders - Overall business performance metrics

print("=" * 80)
print("AGGREGATE 1: TOTAL SALES & ORDERS")
print("=" * 80)
print("\nBusiness Question: What are our overall business performance metrics?")
print("Metrics: Total revenue, orders, quantity, cost, profit, and profit margin")

# Calculate overall totals
total_sales_orders = fact_sales.agg(
    spark_sum("Revenue").alias("TotalRevenue"),
    spark_sum("Cost").alias("TotalCost"),
    spark_sum("Profit").alias("TotalProfit"),
    countDistinct("OrderNumber").alias("TotalOrders"),
    spark_sum("OrderQuantity").alias("TotalQuantitySold"),
    countDistinct("CustomerKey").alias("UniqueCustomers"),
    countDistinct("ProductKey").alias("UniqueProductsSold"),
    avg("Revenue").alias("AvgRevenuePerTransaction"),
    avg("Profit").alias("AvgProfitPerTransaction")
).withColumn(
    "ProfitMarginPct",
    spark_round((col("TotalProfit") / col("TotalRevenue")) * 100, 2)
).withColumn(
    "AvgOrderValue",
    spark_round(col("TotalRevenue") / col("TotalOrders"), 2)
)

print("\n✓ Aggregation complete!")
print("\nOverall Business Performance Metrics:")
display(total_sales_orders)

# Show formatted summary
result = total_sales_orders.first()
print("\n" + "="*80)
print("BUSINESS SUMMARY")
print("="*80)
print(f"\nRevenue Metrics:")
print(f"  Total Revenue:     ${result.TotalRevenue:,.2f}")
print(f"  Total Cost:        ${result.TotalCost:,.2f}")
print(f"  Total Profit:      ${result.TotalProfit:,.2f}")
print(f"  Profit Margin:     {result.ProfitMarginPct}%")
print(f"\nOrder Metrics:")
print(f"  Total Orders:      {result.TotalOrders:,}")
print(f"  Avg Order Value:   ${result.AvgOrderValue:,.2f}")
print(f"  Units Sold:        {result.TotalQuantitySold:,}")
print(f"\nCustomer & Product:")
print(f"  Unique Customers:  {result.UniqueCustomers:,}")
print(f"  Products Sold:     {result.UniqueProductsSold:,}")
print("\n" + "="*80)

AGGREGATE 1: TOTAL SALES & ORDERS

Business Question: What are our overall business performance metrics?
Metrics: Total revenue, orders, quantity, cost, profit, and profit margin

✓ Aggregation complete!

Overall Business Performance Metrics:


TotalRevenue,TotalCost,TotalProfit,TotalOrders,TotalQuantitySold,UniqueCustomers,UniqueProductsSold,AvgRevenuePerTransaction,AvgProfitPerTransaction,ProfitMarginPct,AvgOrderValue
450314774.86,264342354.18,185972420.68,11157,1002366,9373,158,7065.091074,2917.763668,41.30,40361.64



BUSINESS SUMMARY

Revenue Metrics:
  Total Revenue:     $450,314,774.86
  Total Cost:        $264,342,354.18
  Total Profit:      $185,972,420.68
  Profit Margin:     41.30%

Order Metrics:
  Total Orders:      11,157
  Avg Order Value:   $40,361.64
  Units Sold:        1,002,366

Customer & Product:
  Unique Customers:  9,373
  Products Sold:     158



In [0]:
# Aggregate 2: Sales by Month - Track performance over time

print("=" * 80)
print("AGGREGATE 2: SALES BY MONTH")
print("=" * 80)
print("\nBusiness Question: How do sales trend over time? Which months perform best?")
print("Metrics: Revenue, orders, and profit by month")

# Aggregate by year and month
sales_by_month = fact_sales.groupBy(
    col("CalendarYear").alias("Year"),
    col("MonthName").alias("Month"),
    month(col("OrderDate")).alias("MonthNumber")
).agg(
    spark_sum("Revenue").alias("TotalRevenue"),
    spark_sum("Cost").alias("TotalCost"),
    spark_sum("Profit").alias("TotalProfit"),
    countDistinct("OrderNumber").alias("TotalOrders"),
    spark_sum("OrderQuantity").alias("TotalQuantitySold"),
    countDistinct("CustomerKey").alias("UniqueCustomers"),
    avg("Revenue").alias("AvgTransactionValue")
).withColumn(
    "ProfitMarginPct",
    spark_round((col("TotalProfit") / col("TotalRevenue")) * 100, 2)
).orderBy("Year", "MonthNumber")

print("\n✓ Aggregation complete!")
print(f"\nTotal months with data: {sales_by_month.count()}")
print("\nSales by Month (sorted chronologically):")
display(sales_by_month)

# Show top 5 months by revenue
print("\nTop 5 Months by Revenue:")
top_months = sales_by_month.orderBy(col("TotalRevenue").desc()).limit(5)
display(top_months)

print("\n" + "="*80)

AGGREGATE 2: SALES BY MONTH

Business Question: How do sales trend over time? Which months perform best?
Metrics: Revenue, orders, and profit by month

✓ Aggregation complete!

Total months with data: 37

Sales by Month (sorted chronologically):


Year,Month,MonthNumber,TotalRevenue,TotalCost,TotalProfit,TotalOrders,TotalQuantitySold,UniqueCustomers,AvgTransactionValue,ProfitMarginPct
2005,July,7,7861829.92,4707592.06,3154237.86,68,2314,69,56969.782029,40.12
2005,August,8,7294498.56,4325344.40,2969154.16,67,2248,70,52103.561143,40.70
2005,September,9,7808579.94,4693786.68,3114793.26,65,2404,70,55775.571000,39.89
2005,October,10,5150865.02,3075196.52,2075668.50,55,1856,58,44404.008793,40.30
2005,November,11,6753315.30,4035210.06,2718105.24,58,2130,66,51161.479545,40.25
2005,December,12,6300985.78,3784471.40,2516514.38,62,2244,67,47022.281940,39.94
2006,January,1,5201574.72,3101459.34,2100115.38,53,1654,54,48162.728889,40.37
2006,February,2,5399292.80,3233946.94,2165345.86,58,1718,62,43542.683871,40.10
2006,March,3,10969466.00,6575462.62,4394003.38,106,3394,111,49412.009009,40.06
2006,April,4,10669010.58,6402451.08,4266559.50,98,3412,105,50804.812286,39.99



Top 5 Months by Revenue:


Year,Month,MonthNumber,TotalRevenue,TotalCost,TotalProfit,TotalOrders,TotalQuantitySold,UniqueCustomers,AvgTransactionValue,ProfitMarginPct
2008,May,5,30615955.22,17844780.80,12771174.42,1111,90280,1125,5343.098642,41.71
2008,June,6,30354739.26,17756914.64,12597824.62,1086,90028,1098,5279.085089,41.50
2007,December,12,27886412.00,16310649.94,11575762.06,1081,87278,1090,4935.648142,41.51
2008,April,4,26352977.12,15441169.92,10911807.20,1029,84368,1054,4889.235087,41.41
2008,March,3,24948410.38,14468487.22,10479923.16,990,82004,1006,4905.310731,42.01


In [0]:
# Aggregate 3: Sales by Country - Geographic performance analysis

print("=" * 80)
print("AGGREGATE 3: SALES BY COUNTRY")
print("=" * 80)
print("\nBusiness Question: Which countries/regions drive the most revenue?")
print("Metrics: Revenue, orders, and customers by geographic location")

# Aggregate by country and region
sales_by_country = fact_sales.groupBy(
    col("Country"),
    col("Region")
).agg(
    spark_sum("Revenue").alias("TotalRevenue"),
    spark_sum("Cost").alias("TotalCost"),
    spark_sum("Profit").alias("TotalProfit"),
    countDistinct("OrderNumber").alias("TotalOrders"),
    spark_sum("OrderQuantity").alias("TotalQuantitySold"),
    countDistinct("CustomerKey").alias("UniqueCustomers"),
    countDistinct("City").alias("UniqueCities"),
    avg("Revenue").alias("AvgTransactionValue")
).withColumn(
    "ProfitMarginPct",
    spark_round((col("TotalProfit") / col("TotalRevenue")) * 100, 2)
).withColumn(
    "RevenuePerCustomer",
    spark_round(col("TotalRevenue") / col("UniqueCustomers"), 2)
).orderBy(col("TotalRevenue").desc())

print("\n✓ Aggregation complete!")
print(f"\nTotal country-region combinations: {sales_by_country.count()}")
print("\nSales by Country (sorted by revenue):")
display(sales_by_country)

# Country-level summary (roll up regions)
sales_by_country_summary = fact_sales.groupBy("Country").agg(
    spark_sum("Revenue").alias("TotalRevenue"),
    spark_sum("Profit").alias("TotalProfit"),
    countDistinct("OrderNumber").alias("TotalOrders"),
    countDistinct("CustomerKey").alias("UniqueCustomers"),
    countDistinct("Region").alias("RegionCount")
).withColumn(
    "ProfitMarginPct",
    spark_round((col("TotalProfit") / col("TotalRevenue")) * 100, 2)
).orderBy(col("TotalRevenue").desc())

print("\nCountry-Level Summary (Top 10):")
display(sales_by_country_summary.limit(10))

print("\n" + "="*80)

AGGREGATE 3: SALES BY COUNTRY

Business Question: Which countries/regions drive the most revenue?
Metrics: Revenue, orders, and customers by geographic location

✓ Aggregation complete!

Total country-region combinations: 46

Sales by Country (sorted by revenue):


Country,Region,TotalRevenue,TotalCost,TotalProfit,TotalOrders,TotalQuantitySold,UniqueCustomers,UniqueCities,AvgTransactionValue,ProfitMarginPct,RevenuePerCustomer
United States,California,87489699.70,50918120.62,36571579.08,2772,213576,2431,43,6467.304827,41.80,35989.18
United Kingdom,England,59840532.80,35215384.98,24625147.82,1549,116218,969,34,8082.189735,41.15,61754.94
Australia,New South Wales,57235603.18,34045260.00,23190343.18,1454,93626,799,18,9457.303896,40.52,71634.05
United States,Washington,36277750.84,21092019.56,15185731.28,1402,102170,1203,24,5614.012820,41.86,30156.07
Australia,Victoria,33990052.44,20018946.08,13971106.36,848,57884,452,9,9761.646307,41.10,75199.23
Canada,British Columbia,31258825.94,18084002.22,13174823.72,1799,138240,839,16,3474.747214,42.15,37257.24
Australia,Queensland,28075236.26,16608486.54,11466749.72,709,45112,385,8,9871.742707,40.84,72922.69
United States,Oregon,15402509.84,8943076.08,6459433.76,645,47202,553,10,5217.652385,41.94,27852.64
Germany,Hessen,12452333.74,7335139.86,5117193.88,271,19260,174,9,10308.223295,41.09,71565.14
Germany,Saarland,11833106.12,7012021.44,4821084.68,294,20314,195,11,8857.115359,40.74,60682.60



Country-Level Summary (Top 10):


Country,TotalRevenue,TotalProfit,TotalOrders,UniqueCustomers,RegionCount,ProfitMarginPct
United States,139485271.56,58353146.66,4539,4209,16,41.83
Australia,128445760.54,52364300.78,3127,1796,5,40.77
United Kingdom,59840532.80,24625147.82,1549,969,1,41.15
Germany,49020586.88,20132984.14,1193,812,6,41.07
France,41650556.66,17082488.14,1080,740,16,41.01
Canada,31872066.42,13414353.14,1810,847,2,42.09


In [0]:
# Aggregate 4: Sales by Product - Product category and SKU performance

print("=" * 80)
print("AGGREGATE 4: SALES BY PRODUCT")
print("=" * 80)
print("\nBusiness Question: Which products and categories generate the most revenue?")
print("Metrics: Revenue, profit margin, and sales volume by product hierarchy")

# Product Category Level
print("\n--- PRODUCT CATEGORY LEVEL ---")
sales_by_category = fact_sales.groupBy(
    col("ProductCategoryKey"),
    col("ProductCategoryName")
).agg(
    spark_sum("Revenue").alias("TotalRevenue"),
    spark_sum("Cost").alias("TotalCost"),
    spark_sum("Profit").alias("TotalProfit"),
    countDistinct("OrderNumber").alias("TotalOrders"),
    spark_sum("OrderQuantity").alias("TotalQuantitySold"),
    countDistinct("ProductKey").alias("UniqueProducts"),
    avg("Profit").alias("AvgProfitPerTransaction")
).withColumn(
    "ProfitMarginPct",
    spark_round((col("TotalProfit") / col("TotalRevenue")) * 100, 2)
).orderBy(col("TotalRevenue").desc())

print("\n✓ Product Category aggregation complete!")
display(sales_by_category)

# Product Subcategory Level
print("\n--- PRODUCT SUBCATEGORY LEVEL ---")
sales_by_subcategory = fact_sales.groupBy(
    col("ProductCategoryName"),
    col("ProductSubcategoryKey"),
    col("ProductSubcategoryName")
).agg(
    spark_sum("Revenue").alias("TotalRevenue"),
    spark_sum("Profit").alias("TotalProfit"),
    countDistinct("OrderNumber").alias("TotalOrders"),
    spark_sum("OrderQuantity").alias("TotalQuantitySold"),
    countDistinct("ProductKey").alias("UniqueProducts")
).withColumn(
    "ProfitMarginPct",
    spark_round((col("TotalProfit") / col("TotalRevenue")) * 100, 2)
).orderBy(col("TotalRevenue").desc())

print("\n✓ Product Subcategory aggregation complete!")
print("\nTop 15 Subcategories by Revenue:")
display(sales_by_subcategory.limit(15))

# Individual Product Level (SKU)
print("\n--- INDIVIDUAL PRODUCT (SKU) LEVEL ---")
sales_by_product = fact_sales.groupBy(
    col("ProductKey"),
    col("Product_Name"),
    col("ProductSubcategoryName"),
    col("ProductCategoryName"),
    col("Color")
).agg(
    spark_sum("Revenue").alias("TotalRevenue"),
    spark_sum("Profit").alias("TotalProfit"),
    countDistinct("OrderNumber").alias("TotalOrders"),
    spark_sum("OrderQuantity").alias("TotalQuantitySold"),
    avg("List_Price").alias("AvgListPrice")
).withColumn(
    "ProfitMarginPct",
    spark_round((col("TotalProfit") / col("TotalRevenue")) * 100, 2)
).orderBy(col("TotalRevenue").desc())

print("\n✓ Individual Product aggregation complete!")
print(f"\nTotal unique products: {sales_by_product.count()}")
print("\nTop 20 Products by Revenue:")
display(sales_by_product.limit(20))

print("\n" + "="*80)

AGGREGATE 4: SALES BY PRODUCT

Business Question: Which products and categories generate the most revenue?
Metrics: Revenue, profit margin, and sales volume by product hierarchy

--- PRODUCT CATEGORY LEVEL ---

✓ Product Category aggregation complete!


ProductCategoryKey,ProductCategoryName,TotalRevenue,TotalCost,TotalProfit,TotalOrders,TotalQuantitySold,UniqueProducts,AvgProfitPerTransaction,ProfitMarginPct
1,Bikes,432566680.36,256479380.08,176087300.28,6430,234094,116,11830.643663,40.71
4,Accessories,12096739.02,4525497.30,7571241.72,7916,617400,22,193.351083,62.59
3,Clothing,5651355.48,3337476.80,2313878.68,3662,150872,20,238.642603,40.94



--- PRODUCT SUBCATEGORY LEVEL ---

✓ Product Subcategory aggregation complete!

Top 15 Subcategories by Revenue:


ProductCategoryName,ProductSubcategoryKey,ProductSubcategoryName,TotalRevenue,TotalProfit,TotalOrders,TotalQuantitySold,UniqueProducts,ProfitMarginPct
Bikes,2,Road Bikes,219471379.92,83564725.16,3594,124348,60,38.08
Bikes,1,Mountain Bikes,157484514.58,71479459.34,2308,79396,34,45.39
Bikes,3,Touring Bikes,55610785.86,21043115.78,915,30350,22,37.84
Accessories,37,Tires and Tubes,4294646.84,2687869.88,4697,299860,11,62.59
Accessories,31,Helmets,3737281.90,2339139.00,3144,106810,3,62.59
Clothing,21,Jerseys,2777304.52,638903.20,1669,53548,8,23.00
Clothing,22,Shorts,1282216.80,802599.20,582,18320,3,62.59
Accessories,28,Bottles and Cages,948699.80,593541.00,2432,132620,3,62.56
Accessories,30,Fenders,848032.36,530888.32,1168,38582,1,62.60
Accessories,26,Bike Racks,720480.00,451020.48,185,6004,1,62.60



--- INDIVIDUAL PRODUCT (SKU) LEVEL ---

✓ Individual Product aggregation complete!

Total unique products: 158

Top 20 Products by Revenue:


ProductKey,Product_Name,ProductSubcategoryName,ProductCategoryName,Color,TotalRevenue,TotalProfit,TotalOrders,TotalQuantitySold,AvgListPrice,ProfitMarginPct
310,Product 67,Road Bikes,Bikes,Red,17927132.70,7048969.80,139,5010,3578.270000,39.32
363,Product 321,Mountain Bikes,Bikes,Black,17382254.26,7899757.74,231,7574,2294.990000,45.45
353,Product 94,Mountain Bikes,Bikes,Silver,17038006.56,7743293.28,234,7344,2319.990000,45.45
359,Product 317,Mountain Bikes,Bikes,Black,16569827.80,7530532.20,218,7220,2294.990000,45.45
361,Product 319,Mountain Bikes,Bikes,Black,16487208.16,7492983.84,219,7184,2294.990000,45.45
357,Product 98,Mountain Bikes,Bikes,Silver,16267769.88,7393242.44,227,7012,2319.990000,45.45
313,Product 70,Road Bikes,Bikes,Red,15679979.14,6165386.36,124,4382,3578.270000,39.32
355,Product 96,Mountain Bikes,Bikes,Silver,15478973.28,7034756.64,213,6672,2319.990000,45.45
312,Product 69,Road Bikes,Bikes,Red,15357934.84,6038758.16,138,4292,3578.270000,39.32
314,Product 71,Road Bikes,Bikes,Red,13597426.00,5346524.00,122,3800,3578.270000,39.32


In [0]:
# KPI 5: Customer Summary - Customer lifetime value, purchase behavior, and segmentation

print("=" * 80)
print("KPI 5: CUSTOMER SUMMARY - LIFETIME VALUE & BEHAVIOR")
print("=" * 80)
print("\nBusiness Question: Who are our most valuable customers? How do they behave?")
print("Metrics: CLV, purchase frequency, recency, average order value, RFM analysis")

# Calculate customer-level metrics
customer_summary = fact_sales.groupBy(
    col("CustomerKey"),
    col("CustomerName"),
    col("CustomerGender"),
    col("MaritalStatus"),
    col("Country"),
    col("Region"),
    col("City")
).agg(
    # Revenue metrics
    spark_sum("Revenue").alias("LifetimeRevenue"),
    spark_sum("Profit").alias("LifetimeProfit"),
    spark_sum("Cost").alias("LifetimeCost"),
    
    # Order metrics
    countDistinct("OrderNumber").alias("TotalOrders"),
    spark_sum("OrderQuantity").alias("TotalItemsPurchased"),
    avg("Revenue").alias("AvgTransactionValue"),
    
    # Date metrics for recency
    min("OrderDate").alias("FirstPurchaseDate"),
    max("OrderDate").alias("LastPurchaseDate"),
    
    # Product diversity
    countDistinct("ProductKey").alias("UniqueProductsPurchased"),
    countDistinct("ProductCategoryName").alias("UniqueCategoriesPurchased")
).withColumn(
    "AvgOrderValue",
    spark_round(col("LifetimeRevenue") / col("TotalOrders"), 2)
).withColumn(
    "ProfitMarginPct",
    spark_round((col("LifetimeProfit") / col("LifetimeRevenue")) * 100, 2)
).withColumn(
    "CustomerTenureDays",
    datediff(col("LastPurchaseDate"), col("FirstPurchaseDate"))
).withColumn(
    "DaysSinceLastPurchase",
    datediff(current_date(), col("LastPurchaseDate"))
).withColumn(
    "AvgDaysBetweenOrders",
    spark_round(
        when(col("TotalOrders") > 1, 
             col("CustomerTenureDays") / (col("TotalOrders") - 1)
        ).otherwise(None),
        1
    )
)

print("\n✓ Customer summary aggregation complete!")
print(f"\nTotal unique customers analyzed: {customer_summary.count():,}")

# Show top customers by lifetime revenue
print("\nTop 20 Customers by Lifetime Revenue:")
top_customers = customer_summary.orderBy(col("LifetimeRevenue").desc())
display(top_customers.limit(20))

# Customer segmentation summary
print("\n--- CUSTOMER SEGMENTATION ---")

# High-value vs regular customers
customer_segments = customer_summary.select(
    when(col("LifetimeRevenue") >= 10000, "High Value (>$10K)")
    .when(col("LifetimeRevenue") >= 5000, "Medium Value ($5K-$10K)")
    .when(col("LifetimeRevenue") >= 1000, "Regular ($1K-$5K)")
    .otherwise("Low Value (<$1K)").alias("CustomerSegment"),
    col("LifetimeRevenue"),
    col("TotalOrders")
).groupBy("CustomerSegment").agg(
    count("*").alias("CustomerCount"),
    spark_sum("LifetimeRevenue").alias("TotalRevenue"),
    avg("LifetimeRevenue").alias("AvgRevenue"),
    avg("TotalOrders").alias("AvgOrders")
).withColumn(
    "AvgRevenue",
    spark_round(col("AvgRevenue"), 2)
).withColumn(
    "AvgOrders",
    spark_round(col("AvgOrders"), 1)
).orderBy(col("TotalRevenue").desc())

print("\nCustomer Value Segmentation:")
display(customer_segments)

# Purchase frequency analysis
frequency_analysis = customer_summary.select(
    when(col("TotalOrders") >= 20, "Very Frequent (20+)")
    .when(col("TotalOrders") >= 10, "Frequent (10-19)")
    .when(col("TotalOrders") >= 5, "Regular (5-9)")
    .otherwise("Occasional (1-4)").alias("PurchaseFrequency"),
    col("LifetimeRevenue")
).groupBy("PurchaseFrequency").agg(
    count("*").alias("CustomerCount"),
    spark_sum("LifetimeRevenue").alias("TotalRevenue"),
    avg("LifetimeRevenue").alias("AvgRevenue")
).withColumn(
    "AvgRevenue",
    spark_round(col("AvgRevenue"), 2)
).orderBy(col("TotalRevenue").desc())

print("\nPurchase Frequency Analysis:")
display(frequency_analysis)

print("\n" + "="*80)

KPI 5: CUSTOMER SUMMARY - LIFETIME VALUE & BEHAVIOR

Business Question: Who are our most valuable customers? How do they behave?
Metrics: CLV, purchase frequency, recency, average order value, RFM analysis

✓ Customer summary aggregation complete!

Total unique customers analyzed: 9,373

Top 20 Customers by Lifetime Revenue:


CustomerKey,CustomerName,CustomerGender,MaritalStatus,Country,Region,City,LifetimeRevenue,LifetimeProfit,LifetimeCost,TotalOrders,TotalItemsPurchased,AvgTransactionValue,FirstPurchaseDate,LastPurchaseDate,UniqueProductsPurchased,UniqueCategoriesPurchased,AvgOrderValue,ProfitMarginPct,CustomerTenureDays,DaysSinceLastPurchase,AvgDaysBetweenOrders
12131,Customer642,null,M,France,Nord,Dunkerque,528549.30,211640.14,316909.16,5,322,24024.968182,2005-08-24,2008-04-12,11,3,105709.86,40.04,962,6723,240.5
12307,Customer699,null,M,France,Val d'Oise,Cergy,467532.96,180886.50,286646.46,5,340,21251.498182,2005-11-29,2008-05-20,10,3,93506.59,38.69,903,6685,225.8
14207,Customer1247,null,M,Germany,Hessen,München,461665.32,185499.54,276165.78,3,300,38472.110000,2006-03-31,2007-12-19,6,3,153888.44,40.18,628,6838,314.0
14181,Customer6086,null,M,Germany,Hessen,Frankfurt,461495.46,188215.44,273280.02,3,294,46149.546000,2006-02-24,2007-12-30,5,3,153831.82,40.78,674,6827,337.0
12296,Customer694,null,M,France,Essonne,Les Ulis,449194.24,180244.88,268949.36,5,376,18716.426667,2005-11-19,2008-04-27,9,3,89838.85,40.13,890,6708,222.5
11425,Customer5384,null,M,France,Moselle,Metz,428639.90,179812.64,248827.26,6,572,14287.996667,2007-05-26,2008-06-23,15,3,71439.98,41.95,394,6651,78.8
12323,Customer703,null,M,France,Hauts de Seine,Suresnes,421371.40,167185.88,254185.52,4,396,19153.245455,2005-12-25,2007-11-16,11,3,105342.85,39.68,691,6871,230.3
11429,Customer453,null,M,France,Hauts de Seine,Sèvres,419643.00,177516.22,242126.78,6,562,12342.441176,2007-06-17,2008-06-26,14,3,69940.50,42.30,375,6648,75.0
11423,Customer5382,null,M,Germany,Hamburg,Hamburg,417961.92,173214.16,244747.76,4,504,20898.096000,2006-06-12,2008-04-22,10,3,104490.48,41.44,680,6713,226.7
11057,Customer355,null,M,Australia,New South Wales,Lane Cove,417883.52,179157.38,238726.14,3,200,52235.440000,2005-08-25,2007-11-22,4,2,139294.51,42.87,819,6865,409.5



--- CUSTOMER SEGMENTATION ---

Customer Value Segmentation:


CustomerSegment,CustomerCount,TotalRevenue,AvgRevenue,AvgOrders
High Value (>$10K),4155,438981817.52,105651.46,1.9
Regular ($1K-$5K),2777,6879649.14,2477.37,1.3
Medium Value ($5K-$10K),512,3595581.80,7022.62,1.8
Low Value (<$1K),1929,857726.40,444.65,1.1



Purchase Frequency Analysis:


PurchaseFrequency,CustomerCount,TotalRevenue,AvgRevenue
Occasional (1-4),9312,444470369.66,47730.92
Regular (5-9),40,5189949.98,129748.75
Very Frequent (20+),11,401338.94,36485.36
Frequent (10-19),10,253116.28,25311.63


In [0]:
# Save all aggregated tables to workspace.gold schema

print("=" * 80)
print("SAVING AGGREGATED TABLES TO GOLD LAYER")
print("=" * 80)

# Create gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
print("\n✓ Schema workspace.gold ready")

# Save all aggregate tables
print("\n1. Saving gold_total_sales_orders...")
total_sales_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.total_sales_orders")
print(f"   ✓ Saved {total_sales_orders.count():,} rows")

print("\n2. Saving gold_sales_by_month...")
sales_by_month.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.sales_by_month")
print(f"   ✓ Saved {sales_by_month.count():,} rows")

print("\n3. Saving gold_sales_by_country...")
sales_by_country.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.sales_by_country")
print(f"   ✓ Saved {sales_by_country.count():,} rows")

print("\n4. Saving gold_sales_by_country_summary...")
sales_by_country_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.sales_by_country_summary")
print(f"   ✓ Saved {sales_by_country_summary.count():,} rows")

print("\n5. Saving gold_sales_by_product_category...")
sales_by_category.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.sales_by_product_category")
print(f"   ✓ Saved {sales_by_category.count():,} rows")

print("\n6. Saving gold_sales_by_product_subcategory...")
sales_by_subcategory.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.sales_by_product_subcategory")
print(f"   ✓ Saved {sales_by_subcategory.count():,} rows")

print("\n7. Saving gold_sales_by_product...")
sales_by_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.sales_by_product")
print(f"   ✓ Saved {sales_by_product.count():,} rows")

print("\n8. Saving gold_customer_summary...")
customer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.customer_summary")
print(f"   ✓ Saved {customer_summary.count():,} rows")

print("\n" + "=" * 80)
print("\n✓✓✓ ALL GOLD LAYER TABLES SUCCESSFULLY SAVED! ✓✓✓")
print("\n" + "=" * 80)

SAVING AGGREGATED TABLES TO GOLD LAYER

✓ Schema workspace.gold ready

1. Saving gold_total_sales_orders...
   ✓ Saved 1 rows

2. Saving gold_sales_by_month...
   ✓ Saved 37 rows

3. Saving gold_sales_by_country...
   ✓ Saved 46 rows

4. Saving gold_sales_by_country_summary...
   ✓ Saved 6 rows

5. Saving gold_sales_by_product_category...
   ✓ Saved 3 rows

6. Saving gold_sales_by_product_subcategory...
   ✓ Saved 17 rows

7. Saving gold_sales_by_product...
   ✓ Saved 158 rows

8. Saving gold_customer_summary...
   ✓ Saved 9,373 rows


✓✓✓ ALL GOLD LAYER TABLES SUCCESSFULLY SAVED! ✓✓✓



# Gold Layer Summary

## ✅ Aggregated Tables Created

All tables saved to **workspace.gold** schema and optimized for BI/analytics:

### Core Business Metrics

| Table | Description | Use Case |
| --- | --- | --- |
| **total_sales_orders** | Overall business KPIs (1 row) | Executive dashboard, homepage metrics |
| **sales_by_month** | Monthly time series | Trend analysis, seasonality, forecasting |
| **sales_by_country** | Geographic performance by country & region | Regional sales analysis, market penetration |
| **sales_by_country_summary** | Country-level rollup | High-level geographic overview |

### Product Analytics

| Table | Description | Use Case |
| --- | --- | --- |
| **sales_by_product_category** | Top-level category performance | Category comparison, portfolio strategy |
| **sales_by_product_subcategory** | Subcategory drill-down | Product line analysis, merchandising |
| **sales_by_product** | Individual SKU performance | SKU optimization, inventory planning |

### Customer Intelligence

| Table | Description | Use Case |
| --- | --- | --- |
| **customer_summary** | Customer lifetime value & behavior | Customer segmentation, retention, CLV analysis |

## 📊 Key Metrics Available

### Financial Metrics
* **TotalRevenue** - Sum of all sales revenue
* **TotalCost** - Total cost of goods sold
* **TotalProfit** - Net profit (Revenue - Cost)
* **ProfitMarginPct** - Profit as % of revenue
* **AvgOrderValue** - Average transaction size

### Volume Metrics
* **TotalOrders** - Number of unique orders
* **TotalQuantitySold** - Units sold
* **UniqueCustomers** - Distinct customer count
* **UniqueProducts** - Distinct products sold

### Customer Metrics (Customer Summary)
* **LifetimeRevenue** - Total revenue per customer
* **LifetimeProfit** - Total profit per customer
* **TotalOrders** - Purchase frequency
* **AvgTransactionValue** - Average spend per transaction
* **FirstPurchaseDate** / **LastPurchaseDate** - Customer lifecycle
* **DaysSinceLastPurchase** - Recency metric
* **CustomerTenureDays** - Customer age
* **AvgDaysBetweenOrders** - Purchase interval

## 🚀 Quick Usage Examples

### Example 1: Query Total Business Performance
```sql
SELECT * FROM workspace.gold.total_sales_orders
```

### Example 2: Monthly Sales Trend
```sql
SELECT Year, Month, TotalRevenue, TotalOrders, ProfitMarginPct
FROM workspace.gold.sales_by_month
ORDER BY Year, MonthNumber
```

### Example 3: Top 10 Countries by Revenue
```sql
SELECT Country, TotalRevenue, TotalOrders, UniqueCustomers, ProfitMarginPct
FROM workspace.gold.sales_by_country_summary
ORDER BY TotalRevenue DESC
LIMIT 10
```

### Example 4: Best-Selling Product Categories
```sql
SELECT ProductCategoryName, TotalRevenue, TotalQuantitySold, ProfitMarginPct
FROM workspace.gold.sales_by_product_category
ORDER BY TotalRevenue DESC
```

### Example 5: High-Value Customers
```sql
SELECT CustomerName, Country, LifetimeRevenue, TotalOrders, AvgOrderValue
FROM workspace.gold.customer_summary
WHERE LifetimeRevenue >= 10000
ORDER BY LifetimeRevenue DESC
```

## 🎯 Next Steps

1. **Build Dashboards** - Connect BI tools to gold tables for executive dashboards
2. **Schedule Refreshes** - Set up automated data pipeline to refresh gold layer regularly
3. **Add More KPIs** - Extend with additional business metrics as needed:
   * Sales by salesperson/channel
   * Cohort analysis
   * Churn prediction features
   * Inventory turnover metrics
4. **Performance Optimization** - Add partitioning and Z-ordering for large-scale queries